# 02 — Cleaning and Merge

All 4 datasets are cleaned, merged, and saved as `Data/processed/movies_merged.csv`.
All subsequent analysis and ML notebooks read from that single file.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from Source.scripts.clean_data import (
    add_budget_tier,
    add_decade,
    add_roi_column,
    extract_primary_genre,
    extract_year,
    filter_positive_budget_revenue,
    normalize_title,
)
from Source.scripts.helpers import PROCESSED_DATA_DIR, ensure_processed_dir
from Source.scripts.load_data import load_named_dataset

## Step 1 — movies_metadata: Load and Clean

Rows with zero budget or revenue are meaningless for analysis — they are removed.
Genre is extracted from the JSON string and release year is parsed from the date column.

In [2]:
meta = load_named_dataset('the_movies_metadata')
print(f'Raw shape: {meta.shape}')

meta = filter_positive_budget_revenue(meta, 'budget', 'revenue')
print(f'After budget/revenue filter: {meta.shape}')

meta['title_clean']   = normalize_title(meta['title'])
meta['release_year']  = extract_year(meta['release_date'])
meta['primary_genre'] = extract_primary_genre(meta['genres'])

keep = [c for c in [
    'id', 'imdb_id', 'title', 'title_clean', 'budget', 'revenue',
    'runtime', 'release_year', 'primary_genre', 'vote_average', 'vote_count', 'popularity',
] if c in meta.columns]
df = meta[keep].copy()

print(f'Working dataset shape: {df.shape}')
df.head(3)

Raw shape: (45466, 24)
After budget/revenue filter: (5381, 24)
Working dataset shape: (5381, 12)


,id,imdb_id,title,title_clean,budget,revenue,runtime,release_year,primary_genre,vote_average,vote_count,popularity
0,862,tt0114709,Toy Story,toy story,30000000.0,373554033.0,81.0,1995,Animation,7.7,5415.0,21.946943
1,8844,tt0113497,Jumanji,jumanji,65000000.0,262797249.0,104.0,1995,Adventure,6.9,2413.0,17.015539
3,31357,tt0114885,Waiting to Exhale,waiting to exhale,16000000.0,81452156.0,127.0,1995,Comedy,6.1,34.0,3.859495


## Step 2 — TMDB: Add tmdb_popularity

TMDB popularity score is added as an additional feature.
Left join on normalized title — unmatched rows stay NaN.

In [3]:
tmdb = load_named_dataset('tmdb_movies')
tmdb['title_clean'] = normalize_title(tmdb['title'])

if 'popularity' in tmdb.columns:
    tmdb_slim = (
        tmdb[['title_clean', 'popularity']]
        .rename(columns={'popularity': 'tmdb_popularity'})
        .drop_duplicates('title_clean')
    )
    df = df.merge(tmdb_slim, on='title_clean', how='left')
    filled = df['tmdb_popularity'].notna().sum()
    print(f'After TMDB merge: {df.shape} | tmdb_popularity filled: {filled}')
else:
    print('No popularity column found in tmdb_movies')

After TMDB merge: (5381, 13) | tmdb_popularity filled: 3328


## Step 3 — IMDb Ratings: Join via imdb_id

Much more reliable than title matching — especially for foreign-language films.
Uses the imdb_id column that already exists in movies_metadata.

In [4]:
imdb = load_named_dataset('imdb_ratings')
imdb = imdb.rename(columns={
    'tconst': 'imdb_id',
    'averageRating': 'imdb_rating',
    'numVotes': 'imdb_votes',
})

df = df.merge(imdb[['imdb_id', 'imdb_rating', 'imdb_votes']], on='imdb_id', how='left')
print(f'After IMDb merge: {df.shape}')
print(f'imdb_rating filled: {df["imdb_rating"].notna().sum()} / {len(df)}')

After IMDb merge: (5381, 15)
imdb_rating filled: 5379 / 5381


## Step 4 — Rotten Tomatoes: Join via Title

For tomatometer and audience rating.
Join is done on normalized titles — some films may not match, which is expected.

In [5]:
rt = load_named_dataset('rt_movies')
title_col = 'movie_title' if 'movie_title' in rt.columns else 'title'
rt['title_clean'] = normalize_title(rt[title_col])

rt_cols = [c for c in ['title_clean', 'tomatometer_rating', 'audience_rating'] if c in rt.columns]
rt_slim = rt[rt_cols].drop_duplicates('title_clean')

df = df.merge(rt_slim, on='title_clean', how='left')
print(f'After RT merge: {df.shape}')
for col in ['tomatometer_rating', 'audience_rating']:
    if col in df.columns:
        print(f'{col} filled: {df[col].notna().sum()} / {len(df)}')

After RT merge: (5381, 17)
tomatometer_rating filled: 4376 / 5381
audience_rating filled: 4375 / 5381


## Step 5 — Add Derived Columns

ROI, budget tier and decade — used directly in EDA and ML.

In [6]:
df = add_roi_column(df, 'budget', 'revenue')
df = add_budget_tier(df, 'budget')
df = add_decade(df, 'release_year')

print('budget_tier distribution:')
print(df['budget_tier'].value_counts())
print('\ndecade distribution:')
print(df['decade'].value_counts().sort_index())

budget_tier distribution:
budget_tier
Mid            1393
High           1392
Low            1346
Blockbuster    1250
Name: count, dtype: int64

decade distribution:
decade
1910s       4
1920s      17
1930s      38
1940s      45
1950s      75
1960s     130
1970s     190
1980s     520
1990s     942
2000s    1780
2010s    1640
Name: count, dtype: int64


## Step 6 — Null Check and Save

In [7]:
print('Null counts per column:')
print(df.isnull().sum().sort_values(ascending=False))

Null counts per column:
tmdb_popularity       2053
audience_rating       1006
tomatometer_rating    1005
primary_genre           12
imdb_votes               2
imdb_rating              2
runtime                  1
id                       0
popularity               0
budget_tier              0
roi                      0
vote_count               0
imdb_id                  0
vote_average             0
release_year             0
revenue                  0
budget                   0
title_clean              0
title                    0
decade                   0
dtype: int64


In [8]:
ensure_processed_dir()
out_path = PROCESSED_DATA_DIR / 'movies_merged.csv'
df.to_csv(out_path, index=False)
print(f'Saved {len(df)} rows to {out_path}')
df.head(3)

Saved 5381 rows to C:\Users\hadi\Downloads\TeamsDownloads\Data Analiytcs\Data-Avengers-DA-60\Data\processed\movies_merged.csv


,id,imdb_id,title,title_clean,budget,revenue,runtime,release_year,primary_genre,vote_average,vote_count,popularity,tmdb_popularity,imdb_rating,imdb_votes,tomatometer_rating,audience_rating,roi,budget_tier,decade
0,862,tt0114709,Toy Story,toy story,30000000.0,373554033.0,81.0,1995,Animation,7.7,5415.0,21.946943,73.640445,8.3,1172257.0,100.0,92.0,1145.180110,High,1990s
1,8844,tt0113497,Jumanji,jumanji,65000000.0,262797249.0,104.0,1995,Adventure,6.9,2413.0,17.015539,NaN,7.1,412622.0,55.0,62.0,304.303460,Blockbuster,1990s
2,31357,tt0114885,Waiting to Exhale,waiting to exhale,16000000.0,81452156.0,127.0,1995,Comedy,6.1,34.0,3.859495,NaN,6.1,13817.0,56.0,79.0,409.075975,Mid,1990s
